# Workshop Concurrent: Comité de Crisis Hospitalaria (Hospital)

## 🎯 ¿Qué es el Patrón Concurrent?

El patrón **Concurrent** ejecuta **varios agentes en paralelo** con el **mismo input**.
En lugar de encadenar pasos, obtienes **perspectivas simultáneas** (técnica, clínica, seguridad) y las agregas en un único resultado.

Es ideal cuando necesitas **decidir rápido** con puntos de vista especializados.


## 🏥 Escenario del ejercicio (muy concreto)

Se activa un comité de crisis por un incidente operativo con impacto asistencial:
> “ALERTA: caída de la Historia Clínica Electrónica (HCE) en Urgencias (45 min). Hay colas, riesgo de duplicidades y riesgo de errores de medicación; sala de espera llena.”

Lo que queremos demostrar con **Concurrent** es esto:
1. `ResponsableSistemas` propone medidas técnicas inmediatas de continuidad/recuperación.
2. `DireccionMedica` define protocolo asistencial en modo degradado y priorización.
3. `SeguridadPaciente` identifica riesgos y controles para minimizar daños.

### ✅ Qué deberías ver en la salida
- Tres respuestas (una por rol) generadas **en paralelo** con el mismo contexto.
- Un listado de “perspectivas” donde cada bloque está etiquetado por agente.
- Un tiempo total bajo (paralelo) frente a hacerlo secuencial.

---

## Paso 0: Importaciones y configuración

In [1]:
import os
import time
from agent_framework import ChatAgent, ConcurrentBuilder, WorkflowOutputEvent
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print("✅ Entorno cargado y Microsoft Agent Framework importado")

✅ Entorno cargado y Microsoft Agent Framework importado


## Paso 1: Crear tres agentes especializados
Cada uno con su expertise: sistemas, coordinación clínica, seguridad del paciente.

In [2]:
# Agente 1: Sistemas (HCE/EHR)
agente_tech = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_DEPLOYMENT
    ),
    name="ResponsableSistemas",
    instructions="""Eres responsable de Sistemas del hospital.
Analizas impacto técnico y propones acciones inmediatas para continuidad (modo degradado, recuperación, comunicación).
Mantén respuestas concisas (2-3 oraciones) y orientadas a acciones."""
 )

# Agente 2: Dirección Médica / Coordinación clínica
agente_diplomacia = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_DEPLOYMENT
    ),
    name="DireccionMedica",
    instructions="""Eres Dirección Médica coordinando Urgencias.
Prioriza seguridad asistencial, decide protocolos manuales temporales y comunica instrucciones claras al equipo.
Mantén respuestas concisas (2-3 oraciones)."""
 )

# Agente 3: Seguridad del Paciente / Calidad
agente_seguridad = ChatAgent(
    chat_client=OpenAIChatClient(
        base_url=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        model_id=AZURE_OPENAI_DEPLOYMENT
    ),
    name="SeguridadPaciente",
    instructions="""Eres responsable de Seguridad del Paciente.
Identificas riesgos inmediatos (medicación, identificación, duplicidades) y propones controles (doble verificación, checklist, escalado).
Mantén respuestas concisas (2-3 oraciones)."""
 )

print("✅ Tres agentes especialistas creados:")
print("  🖥️  ResponsableSistemas")
print("  🏥  DirecciónMedica")
print("  🛡️  SeguridadPaciente")

✅ Tres agentes especialistas creados:
  🖥️  ResponsableSistemas
  🏥  DirecciónMedica
  🛡️  SeguridadPaciente


In [3]:
# PATRÓN OFICIAL DE MICROSOFT AGENT FRAMEWORK
# ConcurrentBuilder crea un workflow donde:
# - Todos los agentes reciben: el MISMO prompt
# - Todos ejecutan: SIMULTÁNEAMENTE
# - Resultados: se agregan en lista de mensajes

workflow_paralelo = ConcurrentBuilder().participants([
    agente_tech,
    agente_diplomacia,
    agente_seguridad
]).build()

print("✅ Workflow paralelo creado con ConcurrentBuilder")

✅ Workflow paralelo creado con ConcurrentBuilder


## Paso 2: Crear workflow paralelo con ConcurrentBuilder (API oficial de Microsoft Agent Framework)
Con `ConcurrentBuilder`, el framework gestiona automáticamente:
- Ejecución paralela simultánea (todos los agentes reciben el mismo prompt)
- Agregación de resultados
- Optimización de rendimiento

## Paso 3: Ejecutar workflow paralelo
Los tres agentes corren simultáneamente, el framework agrega resultados automáticamente.

In [4]:
async def comite_de_crisis():
    consulta = """ALERTA: caída de la Historia Clínica Electrónica (HCE) en Urgencias.

Situación:
- Tiempo de caída: 45 minutos
- Afiliación/identificación: colas y riesgo de duplicidades
- Medicación: riesgo de errores por falta de registros
- Presión asistencial: alta (sala de espera llena)

Pregunta: ¿Qué plan inmediato propones desde tu rol para las próximas 2 horas?"""
    
    print("\n" + "="*80)
    print("COMITÉ DE CRISIS - EJECUCIÓN PARALELA CON ConcurrentBuilder")
    print("="*80)
    print(f"\n📋 Situación: {consulta}\n")
    print("⏳ Ejecutando 3 agentes EN PARALELO (Microsoft Agent Framework)...\n")
    
    inicio_paralelo = time.time()
    
    # Ejecutar workflow y capturar eventos
    output_evt = None
    async for event in workflow_paralelo.run_stream(consulta):
        if isinstance(event, WorkflowOutputEvent):
            output_evt = event
            break
    
    tiempo_paralelo_total = time.time() - inicio_paralelo
    
    print(f"""✅ Todos los análisis completados en {tiempo_paralelo_total:.2f}s 
          (ejecución paralela)\n""")
    
    if output_evt:
        messages = output_evt.data
        
        print("="*80)
        print("PERSPECTIVAS DE TODOS LOS AGENTES")
        print("="*80 + "\n")
        
        # Mostrar respuestas de todos los agentes
        for i, msg in enumerate(messages, 1): # type: ignore
            author = msg.author_name or ("usuario" if msg.role.value == "user" else "asistente")
            print(f"{i}. [{author}]:")
            print("-" * 80)
            content = msg.text if hasattr(msg, 'text') else str(msg.content)
            print(f"{content}\n")
        
        return {
            "consulta": consulta,
            "messages": messages,
            "tiempo_total": tiempo_paralelo_total
        }

# Ejecutar comité de crisis
resultado = await comite_de_crisis()


COMITÉ DE CRISIS - EJECUCIÓN PARALELA CON ConcurrentBuilder

📋 Situación: ALERTA: caída de la Historia Clínica Electrónica (HCE) en Urgencias.

Situación:
- Tiempo de caída: 45 minutos
- Afiliación/identificación: colas y riesgo de duplicidades
- Medicación: riesgo de errores por falta de registros
- Presión asistencial: alta (sala de espera llena)

Pregunta: ¿Qué plan inmediato propones desde tu rol para las próximas 2 horas?

⏳ Ejecutando 3 agentes EN PARALELO (Microsoft Agent Framework)...

✅ Todos los análisis completados en 6.66s 
          (ejecución paralela)

PERSPECTIVAS DE TODOS LOS AGENTES

1. [usuario]:
--------------------------------------------------------------------------------
ALERTA: caída de la Historia Clínica Electrónica (HCE) en Urgencias.

Situación:
- Tiempo de caída: 45 minutos
- Afiliación/identificación: colas y riesgo de duplicidades
- Medicación: riesgo de errores por falta de registros
- Presión asistencial: alta (sala de espera llena)

Pregunta: ¿Qué pl